In [ ]:
import os
import sys
import json
import glob
import shutil
import numpy as np
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = r"d:\New folder\Non-Invasive\rPPG"
import csv
import time
import math


## Inlined source

The cells below contain the model / loss source that was previously imported from the `neural_methods/` (and `evaluation/`) packages. They are inlined here so the notebook is self-contained.


In [ ]:
# === inlined from neural_methods/model/EfficientPhys.py ===
"""EfficientPhys: Enabling Simple, Fast and Accurate Camera-Based Vitals Measurement
Proceedings of the IEEE/CVF Winter Conference on Applications of Computer Vision (WACV 2023)
Xin Liu, Brial Hill, Ziheng Jiang, Shwetak Patel, Daniel McDuff
"""

import torch
import torch.nn as nn


class Attention_mask(nn.Module):
    def __init__(self):
        super(Attention_mask, self).__init__()

    def forward(self, x):
        xsum = torch.sum(x, dim=2, keepdim=True)
        xsum = torch.sum(xsum, dim=3, keepdim=True)
        xshape = tuple(x.size())
        return x / xsum * xshape[2] * xshape[3] * 0.5

    def get_config(self):
        """May be generated manually. """
        config = super(Attention_mask, self).get_config()
        return config


class TSM(nn.Module):
    def __init__(self, n_segment=10, fold_div=3):
        super(TSM, self).__init__()
        self.n_segment = n_segment
        self.fold_div = fold_div

    def forward(self, x):
        nt, c, h, w = x.size()
        n_batch = nt // self.n_segment
        x = x.view(n_batch, self.n_segment, c, h, w)
        fold = c // self.fold_div
        out = torch.zeros_like(x)
        out[:, :-1, :fold] = x[:, 1:, :fold]  # shift left
        out[:, 1:, fold: 2 * fold] = x[:, :-1, fold: 2 * fold]  # shift right
        out[:, :, 2 * fold:] = x[:, :, 2 * fold:]  # not shift
        return out.view(nt, c, h, w)


class EfficientPhys(nn.Module):

    def __init__(self, in_channels=3, nb_filters1=32, nb_filters2=64, kernel_size=3, dropout_rate1=0.25,
                 dropout_rate2=0.5, pool_size=(2, 2), nb_dense=128, frame_depth=20, img_size=36, channel='raw'):
        super(EfficientPhys, self).__init__()
        self.in_channels = in_channels
        self.kernel_size = kernel_size
        self.dropout_rate1 = dropout_rate1
        self.dropout_rate2 = dropout_rate2
        self.pool_size = pool_size
        self.nb_filters1 = nb_filters1
        self.nb_filters2 = nb_filters2
        self.nb_dense = nb_dense
        # TSM layers
        self.TSM_1 = TSM(n_segment=frame_depth)
        self.TSM_2 = TSM(n_segment=frame_depth)
        self.TSM_3 = TSM(n_segment=frame_depth)
        self.TSM_4 = TSM(n_segment=frame_depth)
        # Motion branch convs
        self.motion_conv1 = nn.Conv2d(self.in_channels, self.nb_filters1, kernel_size=self.kernel_size, padding=(1, 1),
                                  bias=True)
        self.motion_conv2 = nn.Conv2d(self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, bias=True)
        self.motion_conv3 = nn.Conv2d(self.nb_filters1, self.nb_filters2, kernel_size=self.kernel_size, padding=(1, 1),
                                  bias=True)
        self.motion_conv4 = nn.Conv2d(self.nb_filters2, self.nb_filters2, kernel_size=self.kernel_size, bias=True)
        # Attention layers
        self.apperance_att_conv1 = nn.Conv2d(self.nb_filters1, 1, kernel_size=1, padding=(0, 0), bias=True)
        self.attn_mask_1 = Attention_mask()
        self.apperance_att_conv2 = nn.Conv2d(self.nb_filters2, 1, kernel_size=1, padding=(0, 0), bias=True)
        self.attn_mask_2 = Attention_mask()
        # Avg pooling
        self.avg_pooling_1 = nn.AvgPool2d(self.pool_size)
        self.avg_pooling_2 = nn.AvgPool2d(self.pool_size)
        self.avg_pooling_3 = nn.AvgPool2d(self.pool_size)
        # Dropout layers
        self.dropout_1 = nn.Dropout(self.dropout_rate1)
        self.dropout_2 = nn.Dropout(self.dropout_rate1)
        self.dropout_3 = nn.Dropout(self.dropout_rate1)
        self.dropout_4 = nn.Dropout(self.dropout_rate2)
        # Dense layers
        if img_size == 36:
            self.final_dense_1 = nn.Linear(3136, self.nb_dense, bias=True)
        elif img_size == 72:
            self.final_dense_1 = nn.Linear(16384, self.nb_dense, bias=True)
        elif img_size == 96:
            self.final_dense_1 = nn.Linear(30976, self.nb_dense, bias=True)
        else:
            raise Exception('Unsupported image size')
        self.final_dense_2 = nn.Linear(self.nb_dense, 1, bias=True)
        self.batch_norm = nn.BatchNorm2d(3)
        self.channel = channel

    def forward(self, inputs, params=None):
        inputs = torch.diff(inputs, dim=0)
        inputs = self.batch_norm(inputs)

        network_input = self.TSM_1(inputs)
        d1 = torch.tanh(self.motion_conv1(network_input))
        d1 = self.TSM_2(d1)
        d2 = torch.tanh(self.motion_conv2(d1))

        g1 = torch.sigmoid(self.apperance_att_conv1(d2))
        g1 = self.attn_mask_1(g1)
        gated1 = d2 * g1

        d3 = self.avg_pooling_1(gated1)
        d4 = self.dropout_1(d3)

        d4 = self.TSM_3(d4)
        d5 = torch.tanh(self.motion_conv3(d4))
        d5 = self.TSM_4(d5)
        d6 = torch.tanh(self.motion_conv4(d5))

        g2 = torch.sigmoid(self.apperance_att_conv2(d6))
        g2 = self.attn_mask_2(g2)
        gated2 = d6 * g2

        d7 = self.avg_pooling_3(gated2)
        d8 = self.dropout_3(d7)
        d9 = d8.view(d8.size(0), -1)
        d10 = torch.tanh(self.final_dense_1(d9))
        d11 = self.dropout_4(d10)
        out = self.final_dense_2(d11)

        return out


## Training utilities

Seed, train/val split, HR-MAE, best-checkpoint saver, early-stopping, CSV logger.


In [ ]:
# === Training utilities (shared across notebooks) ===
# seed, train/val split, HR-MAE, best-checkpoint saver, early-stopping, CSV logger.
import os
import random as _random
import csv as _csv
import time as _time
import numpy as _np
import torch as _torch
from scipy.signal import periodogram as _periodogram


def set_seed(seed: int = 42):
    """Set seeds for reproducibility across random, numpy, torch (CPU + CUDA)."""
    _random.seed(seed)
    _np.random.seed(seed)
    _torch.manual_seed(seed)
    _torch.cuda.manual_seed_all(seed)
    _torch.backends.cudnn.deterministic = True
    _torch.backends.cudnn.benchmark = False


def train_val_split(dataset, val_ratio: float = 0.2, seed: int = 42):
    """Random split returning (train_subset, val_subset) with a seeded generator."""
    n_total = len(dataset)
    n_val = max(1, int(n_total * val_ratio))
    n_train = n_total - n_val
    g = _torch.Generator().manual_seed(seed)
    return _torch.utils.data.random_split(dataset, [n_train, n_val], generator=g)


def compute_hr_fft(signal_1d, fps: int = 30, lo_hz: float = 0.6, hi_hz: float = 3.3) -> float:
    """Peak-frequency HR (bpm) of a 1-D signal via periodogram, restricted to a band."""
    sig = signal_1d.detach().cpu().numpy() if _torch.is_tensor(signal_1d) else _np.asarray(signal_1d)
    sig = sig.astype(_np.float64).ravel()
    if sig.size < 8 or sig.std() < 1e-8:
        return 0.0
    sig = sig - sig.mean()
    freqs, psd = _periodogram(sig, fs=fps)
    band = (freqs >= lo_hz) & (freqs <= hi_hz)
    if not band.any():
        return 0.0
    return float(freqs[band][psd[band].argmax()] * 60.0)


def compute_hr_mae_batch(preds, labels, fps: int = 30) -> float:
    """Mean absolute HR error (bpm) over a batch.  preds/labels: (N, T) or (T,)."""
    if preds.dim() == 1:
        preds = preds.unsqueeze(0)
    if labels.dim() == 1:
        labels = labels.unsqueeze(0)
    errs = []
    for i in range(preds.shape[0]):
        hr_p = compute_hr_fft(preds[i], fps)
        hr_l = compute_hr_fft(labels[i], fps)
        errs.append(abs(hr_p - hr_l))
    return float(_np.mean(errs)) if errs else 0.0


class BestCheckpointSaver:
    """Save the model's state_dict whenever a tracked metric improves."""
    def __init__(self, path: str, mode: str = "min"):
        assert mode in ("min", "max")
        self.path = path
        self.mode = mode
        self.best = float("inf") if mode == "min" else -float("inf")
        os.makedirs(os.path.dirname(os.path.abspath(path)) or ".", exist_ok=True)

    def step(self, model, metric: float) -> bool:
        improved = (metric < self.best) if self.mode == "min" else (metric > self.best)
        if improved and not (metric != metric):  # reject NaN
            self.best = metric
            _torch.save(model.state_dict(), self.path)
            return True
        return False


class EarlyStopping:
    """Stop training when the tracked metric stops improving for `patience` epochs."""
    def __init__(self, patience: int = 5, mode: str = "min", min_delta: float = 0.0):
        assert mode in ("min", "max")
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.best = float("inf") if mode == "min" else -float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, metric: float) -> bool:
        improved = (
            (self.mode == "min" and metric < self.best - self.min_delta)
            or (self.mode == "max" and metric > self.best + self.min_delta)
        )
        if improved:
            self.best = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


class MetricLogger:
    """Append per-epoch metrics to a CSV.  Creates the file with headers on init."""
    def __init__(self, csv_path: str, fieldnames=None):
        self.csv_path = csv_path
        self.fieldnames = list(fieldnames) if fieldnames else [
            "epoch", "train_loss", "val_loss", "val_hr_mae", "lr", "time_sec"
        ]
        os.makedirs(os.path.dirname(os.path.abspath(csv_path)) or ".", exist_ok=True)
        with open(csv_path, "w", newline="") as f:
            _csv.DictWriter(f, fieldnames=self.fieldnames).writeheader()

    def log(self, **row):
        with open(self.csv_path, "a", newline="") as f:
            _csv.DictWriter(f, fieldnames=self.fieldnames).writerow(
                {k: row.get(k, "") for k in self.fieldnames}
            )


# Seed the run for reproducibility
set_seed(42)
print("Training utils ready  |  seed=42  |  HR-MAE band: 0.6-3.3 Hz (36-198 bpm)")


In [ ]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "data", "Headmotion")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data", "Headmotion", "groupB")

# ----- video / signal params -----
VIDEO_FPS   = 30       # camera frame rate
PPG_FS      = 60       # PPG sensor sampling rate (Hz)

# ----- EfficientPhys preprocessing params -----
CHUNK_LENGTH = 180     # frames per clip
IMG_H, IMG_W = 72, 72
LABEL_TYPE   = "DiffNormalized"  # needs cumsum in post-processing
FRAME_DEPTH  = 10      # TSM temporal window size

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)

In [ ]:
# Read video frames

def read_video_frames(video_path):
    """Read all frames from an MP4 file.

    Returns:
        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)

In [ ]:
def read_ppg_synced(session_path, num_frames):
    """
    Reads the PPG signal from 'ppg.csv' and resamples it to match the exact 
    timestamps of the video frames from 'frame_timestamps.csv'.
    """
    import pandas as pd
    import numpy as np
    import os
    
    # 1. Read the video frame timestamps
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    
    # Validation
    if len(frame_df) != num_frames:
        print(f"Warning: Video has {num_frames} frames, but frame_timestamps.csv has {len(frame_df)} rows. Using min count.")
        min_len = min(len(frame_df), num_frames)
        frame_t = frame_df["timestamp"].values[:min_len]
    else:
        frame_t = frame_df["timestamp"].values

    # 2. Read the raw PPG data
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    
    # Clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    
    # 3. Resample (Interpolate)
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    
    return ppg_resampled.astype(np.float32)

In [ ]:
# Normalization functions

def standardized_data(data):
    """Standardized: global z-score over all pixels and frames."""
    data = data.astype(np.float32)
    m = np.mean(data)
    s = np.std(data)
    if s > 0:
        data = (data - m) / s
    else:
        data = np.zeros_like(data)
    data = np.where(np.isnan(data), np.zeros_like(data), data)
    return data


def diff_normalize_label(label):
    """DiffNormalized label: finite difference normalised by std, zero-padded."""
    diff = np.diff(label.astype(np.float64), axis=0)
    s = np.std(diff)
    if s > 0:
        diff = diff / s
    return np.append(diff, [0.0]).astype(np.float32)

In [ ]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [ ]:
# Discover subjects and read ground truth heart rate

all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_path = subj_dir

    video_pattern = os.path.join(RAW_DATA_PATH, "videos", f"{subj_id}.mkv")
    video_files = glob.glob(video_pattern)
    
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
    })
    print(f"  {subj_id}  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

In [ ]:
# Data preprocessing

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]

    print(f"=== Processing {subj_key} ===")

    # Read video
    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    # Read and sync PPG
    ppg_signal = read_ppg_synced(session_path, T)
    print(f"  PPG green: min={ppg_signal.min():.0f}, max={ppg_signal.max():.0f}")

    # Crop face and resize to 72x72
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)

    # Standardized data (3 channels) -- input for EfficientPhys
    std_data = standardized_data(frames_cropped)  # (T, 72, 72, 3)

    # DiffNormalized label
    label = diff_normalize_label(ppg_signal)  # (T,)

    # Chunk into clips of CHUNK_LENGTH
    clip_num = T // CHUNK_LENGTH
    data_clips  = np.array([std_data[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([label[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH]    for i in range(clip_num)])

    # Save per-subject subfolder
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        np.save(input_path, data_clips[chunk_idx])   # (CHUNK_LENGTH, 72, 72, 3)
        np.save(label_path, label_clips[chunk_idx])  # (CHUNK_LENGTH,)
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*_input*.npy")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

In [ ]:
# PyTorch Dataset + DataLoader

class EfficientPhysDataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (D, H, W, 3)
        label = np.float32(np.load(self.labels[index]))   # (D,)

        # NDHWC -> NDCHW
        data = np.transpose(data, (0, 3, 1, 2))  # (D, 3, H, W)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                      # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]   # +6 skips "_input"

        return data, label, subject_id, chunk_id

# Train/val split (seeded) + DataLoaders
SEED = 42
VAL_RATIO = 0.2
EPOCHS = 30
LR = 1e-4
PATIENCE = 5

SAVE_DIR = os.path.join(REPO_ROOT, "final_model_release")
LOG_PATH = os.path.join(REPO_ROOT, "results/Normal/groupB/train_logs/EfficientPhys.csv")
SAVE_PATH = os.path.join(SAVE_DIR, "GroupB_EfficientPhys.pth")
os.makedirs(SAVE_DIR, exist_ok=True)

train_ds, val_ds = train_val_split(dataset, val_ratio=VAL_RATIO, seed=SEED)
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,
                          num_workers=4, pin_memory=True, generator=g)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)} ({len(train_loader)} batches)  |  Val: {len(val_ds)} ({len(val_loader)} batches)")


In [ ]:
# Optimized training (val/best/early-stop/grad-clip/CSV)
model = EfficientPhys(frame_depth=FRAME_DEPTH, img_size=IMG_H).to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader)
)
saver   = BestCheckpointSaver(SAVE_PATH, mode="min")
stopper = EarlyStopping(patience=PATIENCE, mode="min")
logger  = MetricLogger(LOG_PATH)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}  |  Save: {SAVE_PATH}")


def _forward_efficientphys(batch):
    data, labels = batch[0].to(DEVICE, non_blocking=True), batch[1].to(DEVICE, non_blocking=True)
    N, D, C, H, W = data.shape
    data = data.view(N * D, C, H, W)
    labels = labels.view(-1, 1)
    trim = (N * D) // FRAME_DEPTH * FRAME_DEPTH
    data, labels = data[:trim], labels[:trim]
    # EfficientPhys does torch.diff internally -> append one frame
    data = torch.cat([data, data[-1:].clone()], dim=0)
    pred = model(data)
    return pred, labels, N, trim // N


for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    train_loss_sum, n_train = 0.0, 0
    tbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", ncols=90)
    for batch in tbar:
        optimizer.zero_grad()
        pred, labels, _, _ = _forward_efficientphys(batch)
        loss = criterion(pred, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss_sum += loss.item()
        n_train += 1
        tbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss = train_loss_sum / max(1, n_train)

    model.eval()
    val_loss_sum, val_hr_mae_sum, n_val = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]  ", ncols=90):
            pred, labels, N, Dt = _forward_efficientphys(batch)
            val_loss_sum += criterion(pred, labels).item()
            val_hr_mae_sum += compute_hr_mae_batch(pred.view(N, Dt), labels.view(N, Dt), fps=VIDEO_FPS) * N
            n_val += N
    val_loss   = val_loss_sum / max(1, len(val_loader))
    val_hr_mae = val_hr_mae_sum / max(1, n_val)
    elapsed = time.time() - t0
    cur_lr = optimizer.param_groups[0]["lr"]

    improved = saver.step(model, val_hr_mae)
    marker = " <- best" if improved else ""
    print(f"Epoch {epoch+1:2d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"val_HR_MAE={val_hr_mae:.2f} bpm  lr={cur_lr:.2e}  ({elapsed:.1f}s){marker}")
    logger.log(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss,
               val_hr_mae=val_hr_mae, lr=cur_lr, time_sec=elapsed)

    if stopper.step(val_hr_mae):
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest val HR-MAE: {saver.best:.2f} bpm  ->  {SAVE_PATH}")
